In [1]:
import sys
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "data-pipeline" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import *
from src.features import *
from src.paths import (
    PROJECT_ROOT, MARKET_JSONL, PROCESSED_DIR, ORDERBOOK_DIR, FILTERED_TOKEN_IDS, ensure_dirs
)

ensure_dirs()
print("Using project root:", PROJECT_ROOT)

market_jsonl_path = MARKET_JSONL
markets_path = PROCESSED_DIR / "markets.parquet"
tokens_path = PROCESSED_DIR / "tokens.parquet"
filtered_tokens_path = FILTERED_TOKEN_IDS
raw_orderbook_dir = ORDERBOOK_DIR
feat_orderbook_dir = PROCESSED_DIR / "orderbook"

Using project root: C:\Users\Daron\coding\inefficiency-prediction


In [2]:
# TO READ THE FILTERED TOKEN IDs BACK INTO A SET
relevant_token_ids = set(pd.read_parquet(filtered_tokens_path)["token_id"].astype(str))

In [7]:
# TO READ TOKENS TABLE
tokens_df = pd.read_parquet(tokens_path)
tokens_df = tokens_df[tokens_df["token_id"].isin(relevant_token_ids)]

In [4]:
files = sorted(raw_orderbook_dir.glob("ob_*.parquet"))

if not files:
    print(f"No files found in {raw_orderbook_dir} matching ob_*.parquet")

for i, file_path in enumerate(files, start=1):
    token_stub = file_path.stem.replace("ob_", "")
    output_path = feat_orderbook_dir / f"feat_{token_stub}.parquet"

    print(f"[{i}/{len(files)}] Processing {file_path.name}")
    try:
        feat_df = build_token_feature_table_from_parquet(
            input_path=file_path,
            output_path=output_path,
            depth_n=3,
            drop_json_cols=True,
            token_meta_df=tokens_df
        )
        print(f"Saved {len(feat_df):,} rows -> {output_path.name}")
    except Exception as e:
        print(f"Failed on {file_path.name}: {e}")

print("columns:", feat_df.columns)

[1/140] Processing ob_100280221677614073970031845639470593625091634838451858024140327157791586052991.parquet
Saved 7,620 rows -> feat_100280221677614073970031845639470593625091634838451858024140327157791586052991.parquet
[2/140] Processing ob_101264163544692975734476704096614714686704159151741493681812911782369649789289.parquet
Saved 10,428 rows -> feat_101264163544692975734476704096614714686704159151741493681812911782369649789289.parquet
[3/140] Processing ob_101434403359553929764780234916919166959889590658679054429843672204390513588056.parquet
Saved 59,504 rows -> feat_101434403359553929764780234916919166959889590658679054429843672204390513588056.parquet
[4/140] Processing ob_101564510739658892373078552770639793245135038809362064410922020660639564617759.parquet
Saved 6,546 rows -> feat_101564510739658892373078552770639793245135038809362064410922020660639564617759.parquet
[5/140] Processing ob_102298659433550248987215228870688081194500704214812846837468302147039715413908.parquet
Saved

In [ ]:
import polars as pl

feat_parquets = str(feat_orderbook_dir / "feat_*.parquet")

feat_df = pl.scan_parquet(feat_parquets) 

In [ ]:
feat_df.describe()

statistic,token_id,snapshot_time,snapshot_timestamp_ms,indexed_at_ms,tick_size,min_order_size,orderbook_neg_risk,best_bid,best_bid_size,best_ask,best_ask_size,mid_price,spread,bid_depth_3,ask_depth_3,imbalance_3,bid_depth_total,ask_depth_total,imbalance_total,n_bid_levels,n_ask_levels,market_id,outcome_index,fee_decimal,neg_risk,volume,volume_clob,effective_end_time,time_to_expiry_hours,log_time_to_expiry_hours,delta_best_bid,delta_best_ask,delta_mid_price,seconds_since_prev_snapshot
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64
"""count""","""3322971""","""3322971""",3.322971e6,3.322971e6,3.322971e6,3.322971e6,3.322971e6,3.301422e6,3.322971e6,3.301424e6,3.322971e6,3.279875e6,3.279875e6,3.322971e6,3.322971e6,3.322971e6,3.322971e6,3.322971e6,3.322971e6,3.322971e6,3.322971e6,"""3322971""",3.322971e6,0.0,3.322971e6,3.322971e6,3.322971e6,"""3322971""",3.322971e6,3.322971e6,3.300959e6,3.300961e6,3.279089e6,3.322831e6
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,21549.0,0.0,21547.0,0.0,43096.0,43096.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0,3.322971e6,0.0,0.0,0.0,"""0""",0.0,0.0,22012.0,22010.0,43882.0,140.0
"""mean""",null,"""2025-12-27 17:04:21.108000+00:…",1.7669e12,1.7669e12,0.005298,4.190213,0.463622,0.499867,590635.226256,0.500128,590623.535183,0.499998,0.006812,1.1687e6,1.1686e6,0.000004,4.5637e6,4.5637e6,-0.000001,41.570309,41.570558,null,0.499994,null,0.496714,5.9136e7,5.9136e7,"""2026-01-15 16:52:12.709271+00:…",455.797667,5.237243,0.000004,-0.000004,0.0,163.469521
"""std""",null,null,2.8447e9,2.8447e9,0.004495,1.842059,null,0.399759,1.4398e6,0.399759,1.4398e6,0.398977,0.010966,3.2247e6,3.2246e6,0.54522,5.9279e6,5.9279e6,0.456909,34.207881,34.208056,null,0.5,null,null,6.5338e7,6.5338e7,null,491.822371,1.690083,0.004805,0.004805,0.003656,1162.565684
"""min""","""100280221677614073970031845639…","""2025-10-14 19:45:53.637000+00:…",1.7605e12,1.7605e12,0.001,0.0,0.0,0.001,0.0,0.001,0.0,0.0015,-0.1,0.0,0.0,-1.0,0.0,0.0,-1.0,0.0,0.0,"""1082748""",0.0,null,0.0,8.4080e6,8.4080e6,"""2025-10-30 05:20:53+00:00""",-0.01806,0.0,-0.91,-0.88,-0.535,0.001
"""25%""",null,"""2025-12-07 18:44:07.061000+00:…",1.7651e12,1.7651e12,0.001,5.0,null,0.053,1535.73,0.054,1535.61,0.055,0.001,17894.57,17894.28,-0.458607,366924.5,366924.97,-0.303151,16.0,16.0,null,0.0,null,null,1.0967e7,1.0967e7,"""2026-01-11 04:18:46+00:00""",77.228255,4.359631,0.0,0.0,0.0,2.442
"""50%""",null,"""2026-01-11 07:18:57.142000+00:…",1.7681e12,1.7681e12,0.001,5.0,null,0.5,17738.7,0.5,17738.99,0.5,0.002,96210.11,96211.52,0.000004,1.6274e6,1.6275e6,-3.7052e-7,34.0,34.0,null,0.0,null,null,3.0194e7,3.0194e7,"""2026-01-28 22:52:08+00:00""",311.143505,5.743463,0.0,0.0,0.0,14.348
"""75%""",null,"""2026-01-20 04:05:56.008000+00:…",1.7689e12,1.7689e12,0.01,5.0,null,0.946,322325.9,0.947,322334.01,0.945,0.01,900852.79,900865.0,0.458615,8076713.6,8076870.4,0.303142,61.0,61.0,null,1.0,null,null,1.0121e8,1.0121e8,"""2026-02-01 07:27:48+00:00""",650.193052,6.478806,0.0,0.0,0.0,76.409
"""max""","""993714773641187158294986917407…","""2026-02-04 00:36:05.175000+00:…",1.7702e12,1.7702e12,0.01,5.0,1.0,0.999,2.0062e7,0.999,2.0062e7,0.9985,0.98,4.2740e7,4.2740e7,1.0,1.1344e8,1.1344e8,1.0,197.0,197.0,"""994900""",1.0,null,1.0,2.3507e8,2.3507e8,"""2026-02-04 00:35:28+00:00""",2621.24659,7.871787,0.88,0.91,0.535,256102.767
